「GroupKFoldの検証スコア（R²の平均値）が最も高くなるようなパラメータ」をOptunaが自動探索

In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# 1. データのロードと準備
df_clean = pd.read_csv(r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\01_cleaned_raw.csv")
df_desc = pd.read_csv(r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\02_molecular_descriptors.csv")

X_raw = pd.concat([
    df_clean[['experiment_id', 'humidity', 'temperature', 'material_concentration', 'proportion']].reset_index(drop=True), 
    df_desc
], axis=1)
y_raw = df_clean['wvp'].reset_index(drop=True)

# 欠損値排除と対数変換
valid_idx = X_raw.notna().all(axis=1)
X_valid = X_raw[valid_idx].copy()
y_valid = np.log1p(y_raw[valid_idx])

groups = X_valid['experiment_id'].values
X = X_valid.drop(columns=['experiment_id'])
y = y_valid.values

# 2. 目的関数の定義（GroupKFoldの平均Test R²を最大化する）
def objective(trial):
    # 探索するパラメータの範囲を指定（未知のグループに対して過学習しにくい範囲に調整）
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 7),             # 木の深さを浅めに制限して過学習防止
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),       # 行のサンプリング割合
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9), # 列のサンプリング割合
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'random_state': 42,
        'n_jobs': -1
    }
    
    # 5-Fold グループクロスバリデーション
    gkf = GroupKFold(n_splits=5)
    cv_test_r2 = []
    
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        
        # 予測して元の単位に逆変換
        test_preds_orig = np.expm1(model.predict(X_test))
        y_test_orig = np.expm1(y_test)
        
        cv_test_r2.append(r2_score(y_test_orig, test_preds_orig))
    
    # 5フォールドの平均 Test R² を返す（Optunaはこの値を最大化する）
    return np.mean(cv_test_r2)

# 3. 最適化の実行
if __name__ == "__main__":
    print("--- 🤖 グループ隔離(GroupKFold)対応のOptuna最適化を開始します ---")
    
    # ログを少し静かにする
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    # 評価基準（R²）を最大化（maximize）する設定
    study = optuna.create_study(direction="maximize")
    
    # 50回探索（最大10分で強制打ち切り安全装置付き）
    study.optimize(objective, n_trials=50, timeout=600)
    
    print("\n" + "="*40)
    print("      🏆 最適化が完了しました！      ")
    print("="*40)
    print(f"★ 改善された最高の平均 Test R²: {study.best_value:.4f}")
    print("★ 見つかった最適なパラメータ:")
    for k, v in study.best_params.items():
        print(f"  - {k}: {v}")
    print("="*40)
    
    # ベストパラメータをテキストファイルとして上書き保存
    param_path = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\04_best_params_optuna.txt"
    with open(param_path, "w") as f:
        f.write(str(study.best_params))
    print(f"📁 最適パラメータを '{param_path}' に上書き保存しました。")

--- 🤖 グループ隔離(GroupKFold)対応のOptuna最適化を開始します ---

      🏆 最適化が完了しました！      
★ 改善された最高の平均 Test R²: -0.3562
★ 見つかった最適なパラメータ:
  - n_estimators: 50
  - max_depth: 6
  - learning_rate: 0.010076689218901489
  - subsample: 0.8461816880233577
  - colsample_bytree: 0.6955393671489574
  - min_child_weight: 3
📁 最適パラメータを 'C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\04_best_params_optuna.txt' に上書き保存しました。


wvp<4

In [3]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# 1. データのロードと準備
df_clean = pd.read_csv(r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\wvp_adjustment\01_cleaned_raw.csv")
df_desc = pd.read_csv(r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\wvp_adjustment\02_molecular_descriptors.csv")

X_raw = pd.concat([
    df_clean[['experiment_id', 'humidity', 'temperature', 'material_concentration', 'proportion']].reset_index(drop=True), 
    df_desc
], axis=1)
y_raw = df_clean['wvp'].reset_index(drop=True)

# 欠損値排除と対数変換
valid_idx = X_raw.notna().all(axis=1)
X_valid = X_raw[valid_idx].copy()
y_valid = np.log1p(y_raw[valid_idx])

groups = X_valid['experiment_id'].values
X = X_valid.drop(columns=['experiment_id'])
y = y_valid.values

# 2. 目的関数の定義（GroupKFoldの平均Test R²を最大化する）
def objective(trial):
    # 探索するパラメータの範囲を指定（未知のグループに対して過学習しにくい範囲に調整）
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 7),             # 木の深さを浅めに制限して過学習防止
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),       # 行のサンプリング割合
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9), # 列のサンプリング割合
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'random_state': 42,
        'n_jobs': -1
    }
    
    # 5-Fold グループクロスバリデーション
    gkf = GroupKFold(n_splits=5)
    cv_test_r2 = []
    
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        
        # 予測して元の単位に逆変換
        test_preds_orig = np.expm1(model.predict(X_test))
        y_test_orig = np.expm1(y_test)
        
        cv_test_r2.append(r2_score(y_test_orig, test_preds_orig))
    
    # 5フォールドの平均 Test R² を返す（Optunaはこの値を最大化する）
    return np.mean(cv_test_r2)

# 3. 最適化の実行
if __name__ == "__main__":
    print("--- 🤖 グループ隔離(GroupKFold)対応のOptuna最適化を開始します ---")
    
    # ログを少し静かにする
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    # 評価基準（R²）を最大化（maximize）する設定
    study = optuna.create_study(direction="maximize")
    
    # 50回探索（最大10分で強制打ち切り安全装置付き）
    study.optimize(objective, n_trials=50, timeout=600)
    
    print("\n" + "="*40)
    print("      🏆 最適化が完了しました！      ")
    print("="*40)
    print(f"★ 改善された最高の平均 Test R²: {study.best_value:.4f}")
    print("★ 見つかった最適なパラメータ:")
    for k, v in study.best_params.items():
        print(f"  - {k}: {v}")
    print("="*40)
    
    # ベストパラメータをテキストファイルとして上書き保存
    param_path = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\wvp_adjustment\04_best_params_optuna.txt"
    with open(param_path, "w") as f:
        f.write(str(study.best_params))
    print(f"📁 最適パラメータを '{param_path}' に上書き保存しました。")

--- 🤖 グループ隔離(GroupKFold)対応のOptuna最適化を開始します ---

      🏆 最適化が完了しました！      
★ 改善された最高の平均 Test R²: -0.2318
★ 見つかった最適なパラメータ:
  - n_estimators: 61
  - max_depth: 3
  - learning_rate: 0.011159644226596066
  - subsample: 0.8279434845049781
  - colsample_bytree: 0.6708732442270888
  - min_child_weight: 2
📁 最適パラメータを 'C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\wvp_adjustment\04_best_params_optuna.txt' に上書き保存しました。
